# Fine-tuning LoRA — Assistente Médico (Google Colab + Unsloth)

Fine-tune **Llama 3.2 3B Instruct** com [Unsloth](https://github.com/unslothai/unsloth), exportar **GGUF Q4_K_M** para Ollama e persistir no **Google Drive** e HuggingFace.

**Dataset:** faça upload de `sft_positive_conversations.jsonl` (gerado por `export-positive-conversations.ipynb`) para o runtime do Colab.

**Antes de executar:**
1. **Runtime → Alterar tipo de runtime → GPU** (T4 ou melhor).
2. **Secrets** (ícone de chave): crie `HF_TOKEN` com um token de escrita do [Hugging Face](https://huggingface.co/settings/tokens) — só necessário se for publicar no Hub.
3. Ajuste `DRIVE_EXPORT_BASE` e `SFT_JSONL_PATH` na célula de configuração.


## Configuração (Drive, paths, hiperparâmetros)


In [1]:
from datetime import date
from pathlib import Path

# --- Google Drive (persistência do modelo exportado) ---
MOUNT_DRIVE = True
DRIVE_EXPORT_BASE = Path("/content/drive/MyDrive/assistente-medico/fine-tunes")

# Tag da execução (pasta no Drive)
RUN_TAG = f"assistente-medico-{date.today().isoformat()}"

# Dataset: caminho após upload no Colab (Files → Upload)
SFT_JSONL_PATH = Path("sft_positive_conversations.jsonl")

# None = todas as linhas; frozenset({"generate"}) = só respostas clínicas
TRAIN_CALL_TYPES = None

# Modelo sem quantização para evitar perda de precisão ao exportar em GGUF 4-bit ao final
BASE_MODEL_ID = "unsloth/Llama-3.2-3B-Instruct"

# Colab T4: 4096 costuma caber; reduza para 2048 se OOM
max_seq_length = 9000
FILTER_OVERLONG_EXAMPLES = True

# Treino
TRAIN_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
# MAX_STEPS = 60  # aumente ou use num_train_epochs=1 para treino completo

# Export Ollama (somente Q4_K_M)
GGUF_QUANT = "q4_k_m"

# Hugging Face (opcional)
PUSH_GGUF_TO_HF = True
PUSH_LORA_TO_HF = True
HF_REPO_GGUF = "leanseefeld/assistente-medico-llama32-3b-q4km"
HF_REPO_LORA = "leanseefeld/assistente-medico-llama32-3b-lora"
HF_SECRET_NAME = "HF_TOKEN"


In [2]:
if MOUNT_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

DRIVE_EXPORT_DIR = DRIVE_EXPORT_BASE / RUN_TAG
DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Export no Drive: {DRIVE_EXPORT_DIR}")


Mounted at /content/drive
Export no Drive: /content/drive/MyDrive/assistente-medico/fine-tunes/assistente-medico-2026-05-27


## Instalação do Unsloth (Colab)


In [3]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch

    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {
        "2.10": "0.0.34",
        "2.9": "0.0.33.post1",
        "2.8": "0.0.32.post2",
    }.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


## Carregar modelo base (4-bit) + LoRA


In [4]:
if True: # use caso HuggingFace esteja instável (timeout)
    !pip install modelscope
    import os; os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 82.4 MB/s eta 0:00:00


In [5]:
from unsloth import FastLanguageModel
import torch

dtype = None  # auto: float16 em T4
load_in_4bit = False # False para carregar o modelo sem compressão

hf_token = None
try:
    from google.colab import userdata

    hf_token = userdata.get(HF_SECRET_NAME)
except Exception as exc:
    print(f"HF token não carregado ({exc}). OK se não for publicar no Hub.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=hf_token,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


2026-05-27 15:11:00,842 - modelscope - INFO - Got 11 files, start to download ...


Processing 11 items:   0%|          | 0.00/11.0 [00:00<?, ?it/s]

2026-05-27 15:22:12,188 - modelscope - INFO - Finish downloading 11 files for repo 'unsloth/Llama-3.2-3B-Instruct'
2026-05-27 15:22:12,190 - modelscope - INFO - Creating symbolic link [/root/.cache/modelscope/hub/models/unsloth/Llama-3.2-3B-Instruct].


==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth 2026.5.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## Dataset (`llm_input` + `llm_output`)

Cada linha do export do backend já traz o system prompt da tarefa. O notebook monta o texto SFT com o **chat template** do Llama 3.2.


In [7]:
import json

from datasets import load_dataset

ROLE_ALIASES = {
    "human": "user",
    "ai": "assistant",
    "assistant": "assistant",
    "user": "user",
    "system": "system",
}


def _coerce_llm_input(llm_input) -> list:
    if llm_input is None:
        return []
    if hasattr(llm_input, "tolist"):
        llm_input = llm_input.tolist()
    return list(llm_input)


def normalize_messages(llm_input: list[dict]) -> list[dict]:
    out: list[dict] = []
    for raw in _coerce_llm_input(llm_input):
        msg = raw.to_dict() if hasattr(raw, "to_dict") else raw
        if not isinstance(msg, dict):
            msg = dict(msg)
        role = ROLE_ALIASES.get((msg.get("role") or "").strip().lower(), msg.get("role"))
        content = msg.get("content")
        if content is None:
            continue
        if isinstance(content, list):
            content = json.dumps(content, ensure_ascii=False)
        out.append({"role": role, "content": str(content)})
    return out


def messages_to_sft_text(llm_input, llm_output, *, tok) -> str:
    messages = normalize_messages(llm_input)
    messages.append({"role": "assistant", "content": str(llm_output or "")})
    return tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


def format_sft_row(row):
    return {
        "text": messages_to_sft_text(row["llm_input"], row["llm_output"], tok=tokenizer),
        "call_type": row["call_type"],
    }


if not SFT_JSONL_PATH.is_file():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {SFT_JSONL_PATH}. "
        "Faça upload do JSONL exportado para o runtime do Colab."
    )

raw_dataset = load_dataset("json", data_files=str(SFT_JSONL_PATH))
train_source = raw_dataset["train"]
if TRAIN_CALL_TYPES is not None:
    train_source = train_source.filter(lambda row: row["call_type"] in TRAIN_CALL_TYPES)

print(f"Exemplos para treino: {len(train_source)}")
if len(train_source) == 0:
    raise ValueError("Nenhum exemplo após filtro TRAIN_CALL_TYPES.")

dataset = train_source.map(format_sft_row)

# Auditoria de comprimento
token_lens = [
    len(tokenizer.encode(dataset[i]["text"], add_special_tokens=False))
    for i in range(len(dataset))
]
print(
    f"Tokens/texto: n={len(token_lens)} min={min(token_lens)} "
    f"max={max(token_lens)} média={sum(token_lens) / len(token_lens):.0f}"
)
over = sum(1 for n in token_lens if n > max_seq_length)
if over:
    print(f"  {over} exemplos acima de max_seq_length={max_seq_length}")

if FILTER_OVERLONG_EXAMPLES and over:
    keep = [i for i, n in enumerate(token_lens) if n <= max_seq_length]
    dataset = dataset.select(keep)
    print(f"  Mantidos {len(keep)} exemplos após filtro")

raw_dataset  # noqa: B018 — inspecionar no Colab


Generating train split: 0 examples [00:00, ? examples/s]

Exemplos para treino: 73


Map:   0%|          | 0/73 [00:00<?, ? examples/s]

Tokens/texto: n=73 min=164 max=8630 média=1709


DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'patient_id', 'doctor_id', 'message_id', 'call_type', 'sequence', 'model', 'llm_input', 'llm_output'],
        num_rows: 73
    })
})

## Treino (SFT)


In [8]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    shuffle_dataset=True,
    max_seq_length=max_seq_length,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        warmup_steps=2,
        # max_steps=MAX_STEPS,
        num_train_epochs=1,
        learning_rate=1e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/73 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [9]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB reservados antes do treino.")

trainer_stats = trainer.train()


GPU = Tesla T4. Max memory = 14.563 GB.
6.082 GB reservados antes do treino.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 73 | Num Epochs = 1 | Total steps = 19
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 12,156,928 of 3,224,906,752 (0.38% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.800600
2,2.328900
3,1.877400
4,1.718900
5,1.492400
6,1.951000
7,1.675500
8,1.501700
9,2.032400
10,2.457800


In [10]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"{trainer_stats.metrics['train_runtime']:.1f}s de treino.")
print(f"Pico de VRAM: {used_memory} GB (treino ~{used_memory_for_lora} GB).")


239.1s de treino.
Pico de VRAM: 11.549 GB (treino ~5.467 GB).


## Inferência (testar antes de exportar)

Use prompts no mesmo formato do dataset (`llm_input` com roles). A célula abaixo reutiliza a primeira linha `call_type=generate` do JSONL.


In [11]:
from transformers import TextStreamer


def generate_chat(messages: list[dict], *, max_new_tokens: int = 512) -> str:
    FastLanguageModel.for_inference(model)
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    text_streamer = TextStreamer(tokenizer, skip_prompt=True)
    outputs = model.generate(
        input_ids,
        streamer=text_streamer,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Decodifica só os tokens novos
    new_tokens = outputs[0, input_ids.shape[-1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# Smoke test: primeiro exemplo generate do dataset exportado
generate_rows = raw_dataset["train"].filter(lambda r: r["call_type"] == "generate")
if generate_rows:
    infer_messages = normalize_messages(generate_rows[0]["llm_input"])
    print(f"conversation_id={generate_rows[0].get('conversation_id')}")
else:
    infer_messages = [{"role": "user", "content": "O que é herpes zoster?"}]

print("--- Resposta ---")
_ = generate_chat(infer_messages)


Filter:   0%|          | 0/73 [00:00<?, ? examples/s]

conversation_id=conv_869c9d985cdc
--- Resposta ---


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Herpes zoster é uma infecção viral que afeta o sistema nervoso e pode causar dor, vermelhidão e bolhas na pele. É caracterizada por uma lesão dolorosa e eritematosa que evolui para vesículas, que podem ser dolorosas e causar desconforto. Em geral, a infecção por herpes zoster é causada pelo vírus da varicela-zoster, que também causa a varicela em crianças. A infecção por herpes zoster pode afetar qualquer pessoa, mas é mais comum em pessoas idosas, especialmente aquelas com sistemas nervosos comprometidos.<|eot_id|>


In [12]:
# Teste curto (sem contexto PCDT longo)
short_messages = [
    {
        "role": "system",
        "content": (
            "Você é um assistente clínico de apoio a médicos no Brasil. "
            "Responda em português do Brasil, de forma objetiva."
        ),
    },
    {"role": "user", "content": "Em uma frase: o que é hipotireoidismo subclínico?"},
]
print("--- Resposta (prompt curto) ---")
_ = generate_chat(short_messages, max_new_tokens=256)


--- Resposta (prompt curto) ---
O hipotireoidismo subclínico é um estado em que a glândula tireoide não produz hormônios em quantidades suficientes para causar sintomas óbvios, mas ainda assim pode afetar a saúde do paciente.<|eot_id|>


## Exportar GGUF Q4_K_M + Modelfile (Ollama)

O Unsloth faz merge LoRA → HF → GGUF e grava um **Modelfile** automaticamente (usa `llama.cpp` internamente — não é preciso instalar manualmente).

Artefatos ficam em `DRIVE_EXPORT_DIR`. No Ollama local:

```bash
cd /caminho/para/DRIVE_EXPORT_DIR
ollama create assistente-medico -f Modelfile
ollama run assistente-medico
```


In [13]:
# Export local temporário (Colab tem pouco disco); depois copia para o Drive
LOCAL_EXPORT = Path("export_ollama_q4km")
DRIVE_EXPORT_DIR_GGUF = Path(str(DRIVE_EXPORT_DIR) + "_gguf") # sufixo adicionado automaticamente

print(f"Exportando {GGUF_QUANT} → {LOCAL_EXPORT} (pode levar ~15–25 min)...")
model.save_pretrained_gguf(
    str(LOCAL_EXPORT),
    tokenizer,
    quantization_method=GGUF_QUANT,
)

import shutil

for path in LOCAL_EXPORT.iterdir():
    dest = DRIVE_EXPORT_DIR / path.name
    if path.is_dir():
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(path, dest)
    else:
        shutil.copy2(path, dest)

print(f"Artefatos no Drive:\n  {DRIVE_EXPORT_DIR}")
for p in sorted(DRIVE_EXPORT_DIR.iterdir()):
    size_mb = p.stat().st_size / (1024 * 1024) if p.is_file() else 0
    print(f"  - {p.name}" + (f" ({size_mb:.1f} MB)" if p.is_file() else "/"))

modelfile = DRIVE_EXPORT_DIR_GGUF / "Modelfile"
if modelfile.is_file():
    print("\n--- Modelfile (prévia) ---")
    print(modelfile.read_text(encoding="utf-8")[:1200])


Exportando q4_k_m → export_ollama_q4km (pode levar ~15–25 min)...
Unsloth: Merging model weights to 16-bit format...
Detected local model directory: /root/.cache/modelscope/hub/models/unsloth/Llama-3___2-3B-Instruct
No existing and accessible Hugging Face cache directory found.


Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [00:35<00:35, 35.71s/it]

Copied model-00002-of-00002.safetensors from local model directory


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [07:03<00:00, 211.78s/it]


Copied model-00001-of-00002.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [13:52<00:00, 416.28s/it]


Unsloth: Merge process complete. Saved to `/content/export_ollama_q4km`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['export_ollama_q4km_gguf/Llama-3___2-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsl

In [22]:
LOCAL_EXPORT_GGUF = Path("export_ollama_q4km_gguf")



for path in LOCAL_EXPORT_GGUF.iterdir():
    dest = DRIVE_EXPORT_DIR_GGUF / path.name
    if path.is_dir():
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(path, dest)
    else:
        shutil.copy2(path, dest)

print(f"Artefatos no Drive:\n  {DRIVE_EXPORT_DIR_GGUF}")
for p in sorted(DRIVE_EXPORT_DIR.iterdir()):
    size_mb = p.stat().st_size / (1024 * 1024) if p.is_file() else 0
    print(f"  - {p.name}" + (f" ({size_mb:.1f} MB)" if p.is_file() else "/"))

modelfile = DRIVE_EXPORT_DIR_GGUF / "Modelfile"
if modelfile.is_file():
    print("\n--- Modelfile (prévia) ---")
    print(modelfile.read_text(encoding="utf-8")[:1200])


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/assistente-medico/fine-tunes/assistente-medico-2026-05-27_gguf/Modelfile'

In [25]:
modelfile_content = """FROM ./Llama-3___2-3B-Instruct.Q4_K_M.gguf

TEMPLATE \"\"\"<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{{ .System }}<|eot_id|><|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{{ .Response }}<|eot_id|>\"\"\"

PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|end_of_text|>"
PARAMETER temperature 0.1
"""

output_path = "/content/export_ollama_q4km_gguf/Modelfile"

with open(output_path, "w") as f:
    f.write(modelfile_content)

print(f"Modelfile written to {output_path}")

shutil.copy(
    output_path,
    modelfile
)

Modelfile written to /content/export_ollama_q4km_gguf/Modelfile


PosixPath('/content/drive/MyDrive/assistente-medico/fine-tunes/assistente-medico-2026-05-27_gguf/Modelfile')

## Publicar no Hugging Face (opcional)

Defina `PUSH_GGUF_TO_HF = True` (e/ou `PUSH_LORA_TO_HF = True`) na célula de configuração. O token vem do secret `HF_TOKEN`.

- **GGUF:** publica o modelo quantizado pronto para Ollama, salvo no passo anterior com `model.save_pretrained_gguf`.
- **LoRA:** adapters apenas (menor; requer merge na inferência).


In [26]:
if PUSH_LORA_TO_HF or PUSH_GGUF_TO_HF:
    if not hf_token:
        raise RuntimeError(
            f"Defina o secret Colab '{HF_SECRET_NAME}' para publicar no Hugging Face."
        )

if False:
    print(f"Enviando LoRA → {HF_REPO_LORA}")
    model.push_to_hub(HF_REPO_LORA, token=hf_token)
    tokenizer.push_to_hub(HF_REPO_LORA, token=hf_token)
    print("LoRA publicado.")

if PUSH_GGUF_TO_HF:
    print(f"Enviando GGUF {GGUF_QUANT} → {HF_REPO_GGUF}")

    from huggingface_hub import HfApi, create_repo

    api = HfApi(token=hf_token)
    create_repo(HF_REPO_GGUF, repo_type="model", exist_ok=True, token=hf_token)

    # DRIVE_EXPORT_DIR_GGUF = Path(str(DRIVE_EXPORT_DIR) + "_gguf")

    for local_path in sorted(LOCAL_EXPORT_GGUF.iterdir()):
        print(f"Uploading {local_path.name} ...")
        api.upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=local_path.name,
            repo_id=HF_REPO_GGUF,
            repo_type="model",
            token=hf_token,
        )

    print("GGUF publicado. Use o Modelfile do repositório com Ollama.")

if not (PUSH_LORA_TO_HF or PUSH_GGUF_TO_HF):
    print("Publicação no HF desativada (PUSH_*_TO_HF = False).")


Enviando GGUF q4_k_m → leanseefeld/assistente-medico-llama32-3b-q4km
Uploading Llama-3___2-3B-Instruct.Q4_K_M.gguf ...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...2-3B-Instruct.Q4_K_M.gguf:   1%|1         | 24.0MB / 2.02GB            

Uploading Modelfile ...
GGUF publicado. Use o Modelfile do repositório com Ollama.


## Recuperação após timeout

Esta seção foi criada porque o ambiente do Google Drive foi encerrado antes de a converão e upload pro Hugging Face ser concluída.

Aqui, carregamos o adaptador Lora e tentamos conveter e publicar a quantização em GGUF (compatível com Ollama) novamente.

Execute as células 1, 2, 3 e, opcionalmente, 4, antes de seguir adiante.

In [6]:
if True:
  raise Exception("Execute isto apenas para tentar fazer o upload novamente")

hf_token = None
try:
    from google.colab import userdata

    hf_token = userdata.get(HF_SECRET_NAME)
except Exception as exc:
    print(f"HF token não carregado ({exc}). OK se não for publicar no Hub.")

DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)


In [9]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HF_REPO_LORA,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
    token=hf_token,
)
# Do NOT call get_peft_model() here.
print("Loaded fine-tuned model from Hub:", HF_REPO_LORA)

==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.5.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Loaded fine-tuned model from Hub: leanseefeld/assistente-medico-llama32-3b-lora


In [10]:
print(f"Exporting {GGUF_QUANT} to {DRIVE_EXPORT_DIR} ...")
model.save_pretrained_gguf(
    str(DRIVE_EXPORT_DIR),
    tokenizer,
    quantization_method=GGUF_QUANT,
)

for p in sorted(DRIVE_EXPORT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name} ({p.stat().st_size / 1e9:.2f} GB)")

Exporting q4_k_m to /content/drive/MyDrive/assistente-medico/fine-tunes/assistente-medico-2026-05-26 ...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 3049.29it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [08:12<00:00, 246.28s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/assistente-medico/fine-tunes/assistente-medico-2026-05-26`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/assistente-medico/fine-tunes/assistente-medico-2026-05-26_gguf/

In [18]:
from huggingface_hub import HfApi, create_repo

api = HfApi(token=hf_token)
create_repo(HF_REPO_GGUF, repo_type="model", exist_ok=True, token=hf_token)

DRIVE_EXPORT_DIR_GGUF = Path(str(DRIVE_EXPORT_DIR) + "_gguf")

for local_path in sorted(DRIVE_EXPORT_DIR_GGUF.iterdir()):
    if local_path.suffix.lower() in {".gguf"} or local_path.name == "Modelfile":
        print(f"Uploading {local_path.name} ...")
        api.upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=local_path.name,
            repo_id=HF_REPO_GGUF,
            repo_type="model",
            token=hf_token,
        )

print(f"Done: https://huggingface.co/{HF_REPO_GGUF}")

Uploading Modelfile ...
Uploading llama-3.2-3b-instruct.Q4_K_M.gguf ...
Done: https://huggingface.co/leanseefeld/assistente-medico-llama32-3b-q4km
